# Notebook 03 — SWaT and WADI: cleaning, windows and operating regimes

*Corrected version of the original `02b_swat_wadi_raw_cleaning.ipynb`. It also rebuilds the
operating regimes (`swat_tasks.pkl`, `wadi_tasks.pkl`), whose code was not in the repository.*

**What this notebook does.** It reads the raw SWaT and WADI files, cleans them, downsamples
them by 10, scales them, cuts 30-step windows every 10 steps, and groups the normal windows
into operating regimes that serve as meta-learning tasks.

**Why.** Notebooks 07 to 10 depend on these arrays and regimes. The transfer experiment also
needs the cleaned but **unscaled** data, which the original never saved.

**Input.** `SWaT_Dataset_Normal_v1.xlsx`, `SWaT_Dataset_Attack_v0.xlsx` (SWaT.A1 & A2, Dec 2015)
and `WADI_14days_new.csv`, `WADI_attackdataLABLE.csv` (WADI.A2, 19 Nov 2019), found
automatically. If the arrays saved by the original pipeline are also present
(`swat_normal.npy`, ...), the new arrays are compared with them value by value.

**Output.** `swat_clean.npz`, `wadi_clean.npz` (unscaled and scaled arrays and labels),
`swat_regimes.npz`, `wadi_regimes.npz` and `plant_data_summary.json`. The `.npz` files are
derived from licensed iTrust data, so they are not committed to the repository.

### What was corrected
1. **Label cleaning is checked, not assumed.** The SWaT attack file spells its label three
   ways (`Normal`, `Attack`, `A ttack`). All whitespace is now removed before matching, every
   row must then read `normal` or `attack`, and the attack count is asserted against the
   count of all non-normal spellings (54,621 rows). The original's `!= "Normal"` rule gave
   the right count but would have silently turned any unexpected value into an attack.
2. **WADI's empty and gappy columns are found from the data** and asserted to be the four
   known empty columns, instead of being typed in by hand. The -1 = attack polarity is
   asserted (only 1 and -1 may occur) and the attack count is checked (9,977 rows).
3. **Unscaled downsampled arrays are saved** next to the scaled ones, so the transfer
   notebook can fit the target plant's scaling on the few support windows only.
4. **The regimes are rebuilt with visible code:** per-window feature means and standard
   deviations, standardised, k-means with 10 starts and a fixed seed. k is chosen by the
   silhouette score over k = 8 to 30, the range the original used (as its split files
   show); the curve from k = 2 is also reported, because the original's WADI choice (k = 8)
   sat at the lower edge of its range. Regimes under 80 windows are dropped, as before.
5. **Attack windows are assigned to regimes** (nearest centroid), so the held-out regimes
   can be used for testing. The original defined test regimes but never used them. When the
   rebuilt regimes are identical to the original ones (same k, same regime numbers and
   sizes), the original train / validation / test split of the regimes is used.
6. Window counts are reported next to the earlier ones, together with a stricter label rule
   (a window is anomalous only if at least half of its steps are attacks).

In [1]:
import os, sys

def _find_common():
    """Find maml_common.py: this folder when run locally, /kaggle/input on Kaggle."""
    for root in [os.getcwd(), "/kaggle/input"]:
        if os.path.isdir(root):
            for d, _, files in os.walk(root):
                if "maml_common.py" in files:
                    return d
    raise FileNotFoundError("maml_common.py not found. Run from the Corrected-Notebooks folder, "
                            "or attach that folder to Kaggle as a Dataset.")

sys.path.insert(0, _find_common())
import maml_common as sc
SMOKE = os.environ.get("SMAP_SMOKE") == "1"     # tiny settings for testing only
OUT = sc.output_dir(smoke=SMOKE)
print("shared code:", sc.__file__)
print("outputs go to:", OUT)
print("code version:", sc.git_commit())

import time
import numpy as np, pandas as pd
from sklearn.metrics import silhouette_score
K_GRID_ORIGINAL = list(range(8, 31))
K_GRID_DIAGNOSTIC = list(range(2, 31))
if SMOKE:
    K_GRID_ORIGINAL, K_GRID_DIAGNOSTIC = [8, 9], [2, 8, 9]
paths = {p: [sc.find_data_file(f) for f in files] for p, files in sc.PLANT_FILES.items()}
for p, fs in paths.items():
    print(p, fs)
    assert all(fs), f"raw {p} files not found; set MAML_DATA_ROOT or attach them on Kaggle"

shared code: /home/user/Objective-2/Corrected-Notebooks/maml_common.py
outputs go to: /home/user/Objective-2/Corrected-Notebooks/outputs
code version: 85c9644f9e8f5ee6eae4e65bf032fc8577223ed3
swat ['/home/user/Objective-2/SWAT/SWaT_Dataset_Normal_v1.xlsx', '/home/user/Objective-2/SWAT/SWaT_Dataset_Attack_v0.xlsx']
wadi ['/home/user/Objective-2/WADI/WADI.A2_19 Nov 2019/WADI_14days_new.csv', '/home/user/Objective-2/WADI/WADI.A2_19 Nov 2019/WADI_attackdataLABLE.csv']


## 1 — SWaT: load the two workbooks and clean the labels

Reading the Excel files takes a few minutes.

In [2]:
t0 = time.time()
swat_n_raw, swat_n_lab, swat_sensors = sc.load_swat_workbook(paths["swat"][0])
swat_a_raw, swat_a_lab, swat_sensors_a = sc.load_swat_workbook(paths["swat"][1])
print(f"read in {time.time() - t0:.0f}s: normal {swat_n_raw.shape}, attack {swat_a_raw.shape}")
assert swat_sensors == swat_sensors_a and len(swat_sensors) == sc.SWAT_EXPECTED["sensors"]
assert len(swat_n_raw) == sc.SWAT_EXPECTED["normal_rows"] and len(swat_a_raw) == sc.SWAT_EXPECTED["attack_rows"]
swat_n_attack, n_variants = sc.swat_attack_mask(swat_n_lab)
swat_a_attack, a_variants = sc.swat_attack_mask(swat_a_lab)
print("label spellings in the normal file:", n_variants)
print("label spellings in the attack file:", a_variants)
assert swat_n_attack.sum() == 0, "the normal file contains attack rows"
assert swat_a_attack.sum() == sc.SWAT_EXPECTED["attack_labelled_rows"]
print("attack rows before downsampling:", int(swat_a_attack.sum()))
print("any missing values:", int(np.isnan(swat_n_raw).sum() + np.isnan(swat_a_raw).sum()))

/usr/local/lib/python3.11/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


read in 279s: normal (495000, 51), attack (449919, 51)


label spellings in the normal file: {'Normal': 495000}
label spellings in the attack file: {'Normal': 395298, 'Attack': 54584, 'A ttack': 37}
attack rows before downsampling: 54621
any missing values: 0


## 2 — WADI: drop empty columns, interpolate gaps, remap the label

In [3]:
t0 = time.time()
wadi_n_raw, wadi_a_raw, wadi_a_attack, wadi_sensors, wadi_report = sc.load_wadi(*paths["wadi"])
print(f"read in {time.time() - t0:.0f}s: normal {wadi_n_raw.shape}, attack {wadi_a_raw.shape}")
for k, v in wadi_report.items():
    print(f"  {k}: {v}")
assert len(wadi_sensors) == sc.WADI_EXPECTED["sensors"]
assert len(wadi_n_raw) == sc.WADI_EXPECTED["normal_rows"] and len(wadi_a_raw) == sc.WADI_EXPECTED["attack_rows"]
assert wadi_a_attack.sum() == sc.WADI_EXPECTED["attack_labelled_rows"]
print("attack rows before downsampling:", int(wadi_a_attack.sum()))

read in 50s: normal (784571, 123), attack (172801, 123)
  empty_columns_dropped: ['2_LS_001_AL', '2_LS_002_AL', '2_P_001_STATUS', '2_P_002_STATUS']
  label_column: Attack LABLE (1:No Attack, -1:Attack)
  normal_interpolated_columns: ['1_AIT_002_PV', '1_AIT_004_PV', '2B_AIT_004_PV', '3_AIT_004_PV']
  normal_missing_values_before: 34
  attack_interpolated_columns: []
  attack_missing_values_before: 0
attack rows before downsampling: 9977


## 3 — Downsample by 10 and scale

Values: every 10th row. Labels: a downsampled step is an attack if any of its 10 rows is.
Scaling: MinMax fitted on the normal stream only; attack data clipped to [0, 1]. The
unscaled downsampled arrays are kept as well.

In [4]:
plants = {}
for p, (Xn, Xa, ya, sensors) in {"swat": (swat_n_raw, swat_a_raw, swat_a_attack, swat_sensors),
                                 "wadi": (wadi_n_raw, wadi_a_raw, wadi_a_attack, wadi_sensors)}.items():
    n_u, a_u = sc.downsample_values(Xn), sc.downsample_values(Xa)
    y = sc.downsample_labels_or(ya).astype(np.int8)
    n_s, a_s, scaler = sc.minmax_scale_plant(n_u, a_u)
    plants[p] = {"normal_unscaled": n_u, "attack_unscaled": a_u, "attack_labels": y,
                 "normal_scaled": n_s, "attack_scaled": a_s, "sensors": np.array(sensors), **scaler}
    print(f"{p}: normal {n_u.shape}, attack {a_u.shape}, attack steps {int(y.sum())} "
          f"({y.mean():.3%}), attack values clipped {np.mean((a_s == 0) | (a_s == 1)) - np.mean((n_s == 0) | (n_s == 1)):+.3%} vs normal")

swat: normal (49500, 51), attack (44991, 51), attack steps 5493 (12.209%), attack values clipped +6.008% vs normal


wadi: normal (78457, 123), attack (17280, 123), attack steps 1010 (5.845%), attack values clipped +0.689% vs normal


## 4 — Compare with the arrays of the original pipeline (if available)

The original saved `swat_normal.npy`, `swat_attack.npy`, `swat_attack_labels.npy`,
`wadi_normal.npy`, `wadi_attack.npy` and `wadi_attack_labels.npy`. If they can be found, the
new scaled arrays must match them exactly.

In [5]:
compare = {}
for p in plants:
    for key, name in [("normal_scaled", f"{p}_normal.npy"), ("attack_scaled", f"{p}_attack.npy"),
                      ("attack_labels", f"{p}_attack_labels.npy")]:
        f = sc.find_data_file(name)
        if f is None:
            compare[name] = "old file not available"
            continue
        old = np.load(f)
        same = old.shape == plants[p][key].shape and np.array_equal(old, plants[p][key].astype(old.dtype))
        compare[name] = "identical" if same else f"DIFFERENT (max abs diff {np.abs(old.astype(float) - plants[p][key]).max():.3g})"
for k, v in compare.items():
    print(f"  {k:24s} {v}")

  swat_normal.npy          identical
  swat_attack.npy          identical
  swat_attack_labels.npy   identical
  wadi_normal.npy          identical
  wadi_attack.npy          identical
  wadi_attack_labels.npy   identical


## 5 — Windows and window labels

Length 30, a new window every 10 steps. "any" is the earlier rule (at least one attack step);
"half" is the stricter rule (at least half the steps are attacks).

In [6]:
windows = {}
for p, d in plants.items():
    Wn, n_starts = sc.windows_with_labels(d["normal_scaled"])
    Wa, a_starts, y_any, y_half = sc.windows_with_labels(d["attack_scaled"], d["attack_labels"])
    windows[p] = {"normal": Wn, "attack": Wa, "any": y_any, "half": y_half,
                  "normal_starts": n_starts, "attack_starts": a_starts}
    old = sc.OLD_WINDOW_COUNTS[p]
    print(f"{p}: normal windows {len(Wn)} (earlier {old['normal']}), attack-recording windows {len(Wa)} "
          f"(earlier {old['attack']}), anomalous 'any' {y_any.sum()} (earlier {old['anomalous']}), "
          f"anomalous 'half' {y_half.sum()}")

swat: normal windows 4948 (earlier 4948), attack-recording windows 4497 (earlier 4497), anomalous 'any' 648 (earlier 648), anomalous 'half' 549


wadi: normal windows 7843 (earlier 7843), attack-recording windows 1726 (earlier 1726), anomalous 'any' 140 (earlier 140), anomalous 'half' 102


## 6 — Operating regimes

Each normal window is summarised by its per-feature mean and standard deviation (so 102
numbers for SWaT and 246 for WADI), the summaries are standardised, and k-means is run for
each k. The silhouette score measures how well separated the clusters are (from -1 to 1).
k is chosen as the best silhouette over k = 8 to 30, the original range; the best over
k = 2 to 30 is also shown.

In [7]:
old_splits = {p: (lambda f: __import__("json").load(open(f)) if f else None)(sc.find_data_file(f"{p}_task_splits.json"))
              for p in plants}
regimes = {}
for p in plants:
    t0 = time.time()
    fit = sc.fit_regimes(windows[p]["normal"], K_GRID_DIAGNOSTIC, seed=42, n_init=10)
    curve = {k: f["silhouette"] for k, f in fit["fits"].items()}
    k_best = max(K_GRID_ORIGINAL, key=lambda k: curve[k])
    k_best_any = max(K_GRID_DIAGNOSTIC, key=lambda k: curve[k])
    f = fit["fits"][k_best]
    split = sc.split_regimes(f["labels"], min_size=80, n_val=1, n_test=2, seed=42)
    orig = sc.ORIGINAL_REGIMES[p]
    kept = {g: n for g, n in split["sizes"].items() if g not in split["dropped_small"]}
    same = k_best == orig["k"] and kept == orig["sizes"]
    if same:   # identical regimes: use the original train/val/test split
        split.update({key: orig[key] for key in ["meta_train", "meta_val", "meta_test"]},
                     source="original split (regimes reproduced exactly)")
    else:
        split["source"] = "new seeded split (regimes differ from the original)"
    regimes[p] = {"fit": fit, "k": k_best, "curve": curve, "split": split, "labels": f["labels"],
                  "centroids": f["centroids"]}
    print(f"\n{p}: {time.time() - t0:.0f}s, summary dimensions {fit['summary_dims']}")
    print(f"  chosen k (range 8-30): {k_best}, silhouette {curve[k_best]:.4f}")
    print(f"  best k over 2-30: {k_best_any}, silhouette {curve[k_best_any]:.4f}")
    print("  silhouette by k:", {k: round(v, 3) for k, v in curve.items()})
    print(f"  regime sizes: {split['sizes']} | dropped (<80): {split['dropped_small']}")
    print(f"  regimes identical to the original (same k, ids and sizes): {same}")
    print(f"  split used ({split['source']}): meta-train {split['meta_train']}, meta-val {split['meta_val']}, "
          f"meta-test {split['meta_test']}")
    if old_splits[p]:
        o = old_splits[p]
        print(f"  earlier: k {o['chosen_k']}, silhouette {o['silhouette']:.4f}, kept regime sizes "
              f"{sorted(o['regime_sizes'].values(), reverse=True)}")
        print(f"  now:     kept regime sizes {sorted([v for k, v in split['sizes'].items() if k not in split['dropped_small']], reverse=True)}")


swat: 29s, summary dimensions 102
  chosen k (range 8-30): 12, silhouette 0.3963
  best k over 2-30: 2, silhouette 0.8858
  silhouette by k: {2: 0.886, 3: 0.415, 4: 0.333, 5: 0.332, 6: 0.367, 7: 0.364, 8: 0.38, 9: 0.289, 10: 0.387, 11: 0.308, 12: 0.396, 13: 0.323, 14: 0.339, 15: 0.339, 16: 0.353, 17: 0.362, 18: 0.376, 19: 0.37, 20: 0.372, 21: 0.36, 22: 0.358, 23: 0.357, 24: 0.332, 25: 0.375, 26: 0.347, 27: 0.356, 28: 0.326, 29: 0.352, 30: 0.322}
  regime sizes: {0: 396, 1: 2567, 2: 387, 3: 2, 4: 319, 5: 258, 6: 13, 7: 1, 8: 326, 9: 419, 10: 256, 11: 4} | dropped (<80): [3, 6, 7, 11]
  regimes identical to the original (same k, ids and sizes): True
  split used (original split (regimes reproduced exactly)): meta-train [1, 9, 5, 2, 10], meta-val [0], meta-test [4, 8]
  earlier: k 12, silhouette 0.3987, kept regime sizes [2567, 419, 396, 387, 326, 319, 258, 256]
  now:     kept regime sizes [2567, 419, 396, 387, 326, 319, 258, 256]



wadi: 92s, summary dimensions 246
  chosen k (range 8-30): 8, silhouette 0.1048
  best k over 2-30: 3, silhouette 0.2226
  silhouette by k: {2: 0.218, 3: 0.223, 4: 0.118, 5: 0.103, 6: 0.11, 7: 0.1, 8: 0.105, 9: 0.085, 10: 0.097, 11: 0.084, 12: 0.096, 13: 0.102, 14: 0.086, 15: 0.082, 16: 0.088, 17: 0.086, 18: 0.093, 19: 0.075, 20: 0.095, 21: 0.095, 22: 0.085, 23: 0.078, 24: 0.083, 25: 0.086, 26: 0.089, 27: 0.085, 28: 0.1, 29: 0.094, 30: 0.103}
  regime sizes: {0: 133, 1: 958, 2: 1411, 3: 1925, 4: 623, 5: 971, 6: 1491, 7: 331} | dropped (<80): []
  regimes identical to the original (same k, ids and sizes): True
  split used (original split (regimes reproduced exactly)): meta-train [0, 4, 2, 5, 3], meta-val [6], meta-test [1, 7]
  earlier: k 8, silhouette 0.1009, kept regime sizes [1925, 1491, 1411, 971, 958, 623, 331, 133]
  now:     kept regime sizes [1925, 1491, 1411, 971, 958, 623, 331, 133]


## 7 — Assign attack windows to regimes

Each attack-recording window goes to its nearest regime centroid. A held-out regime can be
tested only if it has at least 20 anomalous and 20 normal attack-recording windows.

In [8]:
for p in plants:
    r = regimes[p]
    r["attack_regime"] = sc.assign_regimes(windows[p]["attack"], r["fit"]["scaler_mean"],
                                           r["fit"]["scaler_scale"], r["centroids"])
    rows = []
    for g in sorted(r["split"]["sizes"]):
        m = r["attack_regime"] == g
        role = next((k for k in ["meta_train", "meta_val", "meta_test"] if g in r["split"][k]), "dropped")
        rows.append({"regime": g, "role": role, "normal_windows": r["split"]["sizes"][g],
                     "attack_windows": int(m.sum()), "anomalous_any": int(windows[p]["any"][m].sum()),
                     "normal_in_attack_file": int((m & (windows[p]["any"] == 0)).sum())})
    df = pd.DataFrame(rows).set_index("regime")
    df["testable"] = (df.anomalous_any >= 20) & (df.normal_in_attack_file >= 20)
    r["table"] = df
    print(f"\n{p}\n{df.to_string()}")


swat
              role  normal_windows  attack_windows  anomalous_any  normal_in_attack_file  testable
regime                                                                                            
0         meta_val             396             407             53                    354      True
1       meta_train            2567            2387            129                   2258      True
2       meta_train             387             251             35                    216      True
3          dropped               2               6              6                      0     False
4        meta_test             319             343             28                    315      True
5       meta_train             258             215             21                    194      True
6          dropped              13             319            319                      0     False
7          dropped               1              14             13                      1     False
8   

## 8 — Save

In [9]:
summary = {"label_spellings": {"swat_attack_file": a_variants, "swat_normal_file": n_variants},
           "wadi_cleaning": wadi_report, "comparison_with_original_arrays": compare}
for p in plants:
    d, w, r = plants[p], windows[p], regimes[p]
    np.savez_compressed(os.path.join(OUT, f"{p}_clean.npz"), **d)
    np.savez_compressed(os.path.join(OUT, f"{p}_regimes.npz"), normal_regime=r["labels"],
                        attack_regime=r["attack_regime"], centroids=r["centroids"],
                        scaler_mean=r["fit"]["scaler_mean"], scaler_scale=r["fit"]["scaler_scale"])
    summary[p] = {"normal_steps": len(d["normal_scaled"]), "attack_steps": len(d["attack_scaled"]),
                  "attack_steps_labelled": int(d["attack_labels"].sum()),
                  "windows": {"normal": len(w["normal"]), "attack": len(w["attack"]),
                              "anomalous_any": int(w["any"].sum()), "anomalous_half": int(w["half"].sum()),
                              "earlier": sc.OLD_WINDOW_COUNTS[p]},
                  "regimes": {"chosen_k": r["k"], "k_range": [8, 30], "silhouette": r["curve"][r["k"]],
                              "silhouette_curve": r["curve"], "split": r["split"],
                              "summary_dims": r["fit"]["summary_dims"],
                              "table": r["table"].reset_index().to_dict(orient="records")}}
print(sc.save_json(os.path.join(OUT, "plant_data_summary.json"), summary))
print(sorted(f for f in os.listdir(OUT) if f.endswith(".npz")))

/home/user/Objective-2/Corrected-Notebooks/outputs/plant_data_summary.json
['swat_clean.npz', 'swat_regimes.npz', 'wadi_clean.npz', 'wadi_regimes.npz']
